# 05_special_token

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch01/05_special_token.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
from collections import defaultdict
import re

In [ ]:
# def count_pairs(ids):
#     counts = defaultdict(int)
#     for pair in zip(ids, ids[1:]):
#         counts[pair] += 1
#     return counts

In [ ]:
def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

In [ ]:
def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

In [ ]:
def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 특수 토큰을 기준으로 분할
    texts = input_text.split(end_token)
    ids_list = [list(text.encode("utf-8")) for text in texts]

    # 기본 어휘(0-255) + 종료 토큰 1개를 제외한 수만큼 병합
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in range(num_merges):
        # 인접 쌍의 빈도 집계
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        # 병합 가능한 쌍이 없으면 루프 종료
        if not counts:
            break

        # 가장 빈번한 토큰 쌍 선택
        best_pair = max(counts, key=counts.get)
        # best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 병합
        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules

In [ ]:
# 사용 예
sample_text = "Hello world!<|endoftext|>This is BPE training."

In [ ]:
merge_rules = train_bpe(sample_text, vocab_size=260)
print(merge_rules)  # {(105, 115): 256, (256, 32): 257, (105, 110): 258}

In [ ]:
class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                ids = self._encode_text(text)
                all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

In [ ]:
tokenizer = BPETokenizer(merge_rules)

In [ ]:
text = "Hello world!<|endoftext|>"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

In [ ]:
print(ids)
print(decoded)